# Day 8 — Artificial Neural Network (ANN)

**Goal:** train a small Keras classifier; compare test metrics to Day 7.

> **Read first:** [CONCEPTS.md](./CONCEPTS.md)

## Setup

Run with cwd `notebooks/day08/`. Logic: `src/ann_models.py`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "dataset").is_dir() and (p / "requirements.txt").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sns.set_theme(style="whitegrid", context="notebook")

from sklearn.model_selection import train_test_split

from src.ann_models import build_ann, evaluate_ann, plot_training_history, save_day08_artifacts, train_ann
from src.preprocessing import load_cleaned_table, prepare_train_test_bundle, save_artifacts


## 1. Load Day 3 features

In [ ]:
df = load_cleaned_table(PROJECT_ROOT)
bundle = prepare_train_test_bundle(df, test_size=0.2, random_state=42)
save_artifacts(bundle, PROJECT_ROOT)
X_train, X_test = bundle["X_train"], bundle["X_test"]
y_train, y_test = bundle["y_train"], bundle["y_test"]
class_names = bundle["career_classes"]

## 2. Train / validation split (from train only)

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train,
)
model = build_ann(X_tr.shape[1], len(class_names))
history = train_ann(model, X_tr, y_tr, X_val, y_val)
fig = plot_training_history(history)
plt.show()

## 3. Test evaluation + save

In [ ]:
metrics = evaluate_ann(model, X_test, y_test, class_names)
print(metrics["classification_report"])
paths = save_day08_artifacts(model, history, metrics, class_names, PROJECT_ROOT)
paths

## Checkpoint

Softmax outputs a **probability distribution** over careers — suitable for multi-class classification and Day 10 ranking.